# NILM-Engine — 아키텍처 비교 실험 (Baseline Comparison)

**변수 통제 원칙**: 아키텍처 외 모든 학습 조건을 동일하게 고정합니다.

| 조건 | 값 |
|------|----|
| learning_rate | 1e-3 |
| batch_size | **64** (전 모델 동일) |
| optimizer | Adam (weight_decay=1e-4) |
| scheduler | ReduceLROnPlateau (factor=0.5, patience=2) |
| appliance_loss_scale | **없음** (전 모델 동일) |
| epochs / patience | 15 / 5 |

> **유일한 차이**: 모델 아키텍처 (seq2point · bert4nilm · cnn_tda)  
> 결과 차이 = 순수 아키텍처 차이

## 1. 환경 설치 & GCS 인증

In [ ]:
!pip install -q gudhi

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("GCS 인증 완료")

## 2. 경로 & 분할 상수

In [ ]:
SPLIT = {
    "train": ["house_011", "house_015", "house_016", "house_017",
              "house_033", "house_039", "house_054", "house_063"],
    "val":   ["house_049"],
    "test":  ["house_067"],
}
BUCKET_PREFIX = "ax-nilm-data-dhwang0803-us/nilm/training_dev10"
LABEL_PATH    = "ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet"
EXP_NAME      = "EXP1"   # 기본값 비교는 week-1 단일 실험으로 충분

print(f"Train {len(SPLIT['train'])}개 / Val {len(SPLIT['val'])}개")

## 3. 리포지토리 설정

In [ ]:
import os, sys

REPO_DIR = "/content/ax_nilm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhwang0803-glitch/ax_nilm {REPO_DIR} -q
    print("클론 완료")
else:
    !git -C {REPO_DIR} pull -q && echo "최신화 완료"

SRC_DIR     = f"{REPO_DIR}/nilm-engine/src"
SCRIPTS_DIR = f"{REPO_DIR}/nilm-engine/scripts"
CFG_DIR     = f"{REPO_DIR}/nilm-engine/config"
for d in [SRC_DIR, SCRIPTS_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)
print("sys.path 설정 완료")

## 4. 통일 학습 조건 정의

아키텍처 외 모든 변수를 동일하게 고정합니다.  
batch_size=64는 bert4nilm 메모리 제약 기준으로 전 모델에 동일 적용합니다.

In [ ]:
BASELINE_CFG = {
    "seq2point": {
        "lr":              1e-3,
        "batch_size":      64,
        "appliance_scale": {},
        "description":     "seq2point 원논문(Zhang 2018) 기본값",
    },
    "bert4nilm": {
        "lr":              1e-3,
        "batch_size":      64,
        "appliance_scale": {},
        "description":     "BERT4NILM — 변수 통제 기준 (lr·batch 동일)",
    },
    "cnn_tda": {
        "lr":              1e-3,
        "batch_size":      64,
        "appliance_scale": {},
        "description":     "CNN+TDA — 변수 통제 기준 (lr·batch 동일)",
    },
}

print("통일 학습 조건:")
print(f"  lr=1e-3  batch=64  appliance_scale=없음")
print()
for name, cfg in BASELINE_CFG.items():
    print(f"  {name:14s}  {cfg['description']}")

## 5. Import & 설정 로드

In [ ]:
import json, time, gc
import yaml
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

import gcsfs
gcs = gcsfs.GCSFileSystem()

from acquisition.gcs_loader import GCSNILMDataset
from train_model import (
    build_model, evaluate, train_one_epoch,
    compute_pos_weight, _NILMDatasetWithTDA, APPLIANCE_LABELS,
)

with open(f"{CFG_DIR}/train.yaml")   as f: TRAIN_CFG   = yaml.safe_load(f)
with open(f"{CFG_DIR}/dataset.yaml") as f: DATASET_CFG = yaml.safe_load(f)

print("imports 완료")
print(f"공통 설정: epochs={TRAIN_CFG['training']['epochs']}  "
      f"patience={TRAIN_CFG['training']['early_stopping_patience']}  "
      f"lambda_mse={TRAIN_CFG['training']['loss_weights']['mse']}")

## 6. Google Drive 마운트 (체크포인트 & 결과 저장)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# 비교 실험 전용 폴더 — 기존 EXP1~4 결과와 분리
BASE_DIR    = Path("/content/drive/MyDrive/nilm/arch_compare")
CKPT_DIR    = BASE_DIR / "checkpoints"
RESULTS_DIR = BASE_DIR / "results"
CACHE_DIR   = Path("/content/drive/MyDrive/nilm/cache")   # 캐시는 공유

for d in [CKPT_DIR, RESULTS_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("저장 경로:")
print(f"  체크포인트 → {CKPT_DIR}")
print(f"  결과/그래프 → {RESULTS_DIR}")
print(f"  캐시 (공유) → {CACHE_DIR}")
print()
print("생성 파일 목록 (예시):")
for m in ["seq2point", "bert4nilm", "cnn_tda"]:
    print(f"  {RESULTS_DIR}/EXP1_{m}_baseline_metrics.json")
    print(f"  {CKPT_DIR}/EXP1_{m}_baseline.pt")

## 7. 데이터 캐시 빌드 (EXP1, CPU 런타임 권장)

> GPU 전환 전에 CPU 런타임으로 먼저 실행하면 학습 시간이 단축됩니다.

In [ ]:
ws = DATASET_CFG["window"]["size"]
st = DATASET_CFG["window"]["stride"]
ec = DATASET_CFG["window"].get("event_context")
ss = DATASET_CFG["window"].get("steady_stride")
exp_cfg = TRAIN_CFG["experiments"][EXP_NAME]
week    = exp_cfg.get("week")

for split_name, houses, split_week in [
    ("train", SPLIT["train"], week),
    ("val",   SPLIT["val"],   None),
]:
    print(f"\n[캐시 빌드] {split_name}  week={split_week}")
    _ds = GCSNILMDataset(
        houses, gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=st,
        week=split_week,
        fit_scaler=(split_name == "train"),
        cache_dir=CACHE_DIR,
        event_context=ec, steady_stride=ss,
    )
    print(f"  완료: {len(_ds):,} windows")
    del _ds; gc.collect()

## 8. run_baseline 함수

In [ ]:
def run_baseline(model_name: str, exp_name: str = EXP_NAME,
                 force: bool = False) -> dict:
    """모델별 기본 하이퍼파라미터로 학습. Drive에 저장하고 metrics 반환.

    force=True: 이미 저장된 결과가 있어도 재학습.
    """
    metrics_path = RESULTS_DIR / f"{exp_name}_{model_name}_baseline_metrics.json"
    if metrics_path.exists() and not force:
        print(f"[{model_name}] 이미 완료 — 저장된 결과 로드")
        return json.load(open(metrics_path))

    cfg = BASELINE_CFG[model_name]
    print(f"\n{'='*60}")
    print(f"  {model_name}  —  {cfg['description']}")
    print(f"  lr={cfg['lr']:.0e}  batch={cfg['batch_size']}")
    print(f"{'='*60}")

    exp_cfg = TRAIN_CFG["experiments"][exp_name]
    ws      = DATASET_CFG["window"]["size"]
    stride  = DATASET_CFG["window"]["stride"]
    ec      = DATASET_CFG["window"].get("event_context")
    ss      = DATASET_CFG["window"].get("steady_stride")

    # ── 데이터셋 ─────────────────────────────────────────────────────────────
    base_train = GCSNILMDataset(
        SPLIT["train"], gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=stride,
        week=exp_cfg.get("week"),
        fit_scaler=True,
        cache_dir=CACHE_DIR,
        event_context=ec, steady_stride=ss,
    )
    base_val = GCSNILMDataset(
        SPLIT["val"], gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=stride,
        scaler=base_train.scaler,
        cache_dir=CACHE_DIR,
        event_context=ec, steady_stride=ss,
    )

    if model_name == "cnn_tda":
        train_ds = _NILMDatasetWithTDA(base_train, cache_dir=CACHE_DIR,
                                        event_context=ec, steady_stride=ss)
        val_ds   = _NILMDatasetWithTDA(base_val,   cache_dir=CACHE_DIR,
                                        event_context=ec, steady_stride=ss)
    else:
        train_ds, val_ds = base_train, base_val

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"],
                               shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch_size"],
                               shuffle=False, num_workers=2, pin_memory=True)
    print(f"  train={len(train_ds):,}  val={len(val_ds):,} windows")

    # ── 모델 ─────────────────────────────────────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  device: {device}")
    model = build_model(model_name, ws).to(device)

    # appliance loss scale — 모델별 독립 설정 (seq2point/bert4nilm은 {} → 전부 1.0)
    _app_idx  = {n: i for i, n in enumerate(APPLIANCE_LABELS)}
    app_scale = torch.ones(len(APPLIANCE_LABELS), device=device)
    for aname, s in cfg.get("appliance_scale", {}).items():
        if aname in _app_idx:
            app_scale[_app_idx[aname]] = float(s)
            print(f"  appliance_scale [{aname}]: ×{s}")

    pos_weight = None
    if model_name == "cnn_tda":
        print("  pos_weight 계산 중...")
        pos_weight = compute_pos_weight(train_loader, device, max_weight=50.0)

    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=cfg["lr"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2)

    epochs   = TRAIN_CFG["training"]["epochs"]
    patience = TRAIN_CFG["training"]["early_stopping_patience"]
    lmse     = TRAIN_CFG["training"]["loss_weights"]["mse"]

    best_score = (-float("inf"), float("inf"))
    best_state = None
    no_improve = 0
    history    = []
    amp_scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    # ── 학습 루프 ─────────────────────────────────────────────────────────────
    print(f"  [학습 시작] epochs={epochs}  patience={patience}")
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tloss = train_one_epoch(
            model, train_loader, optimizer, model_name, device,
            amp_scaler, pos_weight=pos_weight,
            lambda_mse=lmse, appliance_scale=app_scale,
        )
        vm = evaluate(model, val_loader, model_name, device)
        scheduler.step(vm["mae"])
        lr_now = optimizer.param_groups[0]["lr"]

        f1c_str = (f"  f1_cls={vm['f1_cls']:.3f}" if vm.get("f1_cls") else "")
        print(f"  ep {epoch:3d}/{epochs}  loss={tloss:.4f}  "
              f"mae={vm['mae']:.2f}W  f1={vm['f1']:.3f}{f1c_str}  lr={lr_now:.2e}")

        history.append({"epoch": epoch, "train_loss": tloss,
                         "val_mae": vm["mae"], "val_f1": vm["f1"]})

        _f1c   = vm.get("f1_cls") or 0.0
        _score = (_f1c, -vm["mae"])
        if _score > best_score or best_state is None:
            best_score = _score
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            no_improve  = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  조기 종료 ({patience} epoch 개선 없음)")
                break

    # ── 최종 평가 & 저장 ──────────────────────────────────────────────────────
    elapsed = time.time() - t0
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    final_m = evaluate(model, val_loader, model_name, device)
    final_m["train_sec"] = int(elapsed)

    ckpt_path = CKPT_DIR / f"{exp_name}_{model_name}_baseline.pt"
    torch.save({"model_state": best_state, "model_name": model_name}, ckpt_path)

    metrics_path.write_text(json.dumps(final_m, ensure_ascii=False, indent=2))

    hist_path = RESULTS_DIR / f"{exp_name}_{model_name}_baseline_history.json"
    hist_path.write_text(json.dumps(history, ensure_ascii=False))

    f1c_final = f"  F1_cls={final_m['f1_cls']:.3f}" if final_m.get("f1_cls") else ""
    print(f"\n  ✅ {model_name}:  MAE={final_m['mae']:.2f}W  "
          f"RMSE={final_m['rmse']:.2f}W  F1={final_m['f1']:.3f}{f1c_final}  ({elapsed:.0f}s)")
    return final_m

## 9. 학습 실행 (3개 모델)

> 순서: seq2point → bert4nilm → cnn_tda  
> 예상 시간: ~20min / ~45min / ~60min (A100 GPU 기준)  
> 중단 후 재시작해도 완료된 모델은 Drive에서 자동 복원됩니다.

In [ ]:
RESULTS = {}
for model_name in ["seq2point", "bert4nilm", "cnn_tda"]:
    RESULTS[model_name] = run_baseline(model_name)

## 10. 결과 비교

In [ ]:
# 런타임 재시작 후 Drive에서 결과 복원
if "RESULTS" not in globals():
    RESULTS = {}
for mname in ["seq2point", "bert4nilm", "cnn_tda"]:
    p = RESULTS_DIR / f"EXP1_{mname}_baseline_metrics.json"
    if p.exists() and mname not in RESULTS:
        RESULTS[mname] = json.load(open(p))
        print(f"  복원: {mname}")

### 10-1. 지표 테이블

In [ ]:
print(f"{'모델':14s} {'lr':>8s} {'MAE(W)':>8s} {'RMSE(W)':>9s} "
      f"{'SAE':>7s} {'F1':>6s} {'F1_cls':>8s} {'시간':>7s}")
print("-" * 72)

for mname in ["seq2point", "bert4nilm", "cnn_tda"]:
    m = RESULTS.get(mname)
    if not m:
        print(f"{mname:14s}  (결과 없음)")
        continue
    cfg  = BASELINE_CFG[mname]
    f1c  = f"{m['f1_cls']:.3f}" if m.get("f1_cls") else "—"
    tsec = f"{m.get('train_sec', 0)//60}m{m.get('train_sec', 0)%60:02d}s"
    print(f"{mname:14s} {cfg['lr']:>8.0e}  {m['mae']:>8.2f}  {m['rmse']:>9.2f}  "
          f"{m['sae']:>7.4f}  {m['f1']:>6.3f}  {f1c:>8s}  {tsec:>7s}")

### 10-2. F1 & MAE 바 차트

In [ ]:
import matplotlib.pyplot as plt

models  = ["seq2point", "bert4nilm", "cnn_tda"]
labels  = MODELS
colors  = ["steelblue", "darkorange", "forestgreen"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# F1
ax = axes[0]
f1_vals = [RESULTS[m]["f1"] if RESULTS.get(m) else 0 for m in models]
bars = ax.bar(labels, f1_vals, color=colors, width=0.5, edgecolor="white")
for bar, v in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
            f"{v:.3f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.set_title("Val F1 (weighted macro)", fontsize=13)
ax.set_ylabel("F1")
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.4, linewidth=1)
ax.grid(axis="y", alpha=0.3)

# MAE
ax = axes[1]
mae_vals = [RESULTS[m]["mae"] if RESULTS.get(m) else 0 for m in models]
bars = ax.bar(labels, mae_vals, color=colors, width=0.5, edgecolor="white")
for bar, v in zip(bars, mae_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f"{v:.1f}W", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_title("Val MAE — 낮을수록 좋음", fontsize=13)
ax.set_ylabel("MAE (W)")
ax.grid(axis="y", alpha=0.3)

plt.suptitle("EXP1 아키텍처 비교 (lr=1e-3 · batch=64 · 동일 조건)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_compare.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {RESULTS_DIR}/baseline_compare.png")

### 10-3. 학습 곡선 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for mname, color in zip(models, colors):
    hist_path = RESULTS_DIR / f"EXP1_{mname}_baseline_history.json"
    if not hist_path.exists():
        continue
    hist = json.load(open(hist_path))
    ep   = [h["epoch"] for h in hist]
    axes[0].plot(ep, [h["train_loss"] for h in hist],
                  label=mname, color=color, marker="o", markersize=4, linewidth=2)
    axes[1].plot(ep, [h["val_f1"]    for h in hist],
                  label=mname, color=color, marker="o", markersize=4, linewidth=2)

for ax, title, ylabel in zip(
    axes,
    ["Train Loss per Epoch", "Val F1 per Epoch"],
    ["Loss", "F1"],
):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 10-4. Per-Appliance F1 비교

In [ ]:
pa_f1 = {}
for mname in models:
    m = RESULTS.get(mname)
    if m and m.get("per_appliance"):
        for app, mets in m["per_appliance"].items():
            pa_f1.setdefault(app, {})[mname] = mets.get("f1")

# 3 모델 모두 F1이 존재하는 가전만 (val positive 있는 가전)
valid_apps = sorted(
    [a for a in pa_f1 if all(pa_f1[a].get(m) is not None for m in models)],
    key=lambda a: -np.mean([pa_f1[a][m] for m in models]),
)

if not valid_apps:
    print("per_appliance 데이터 없음 — 학습 완료 후 재실행하세요.")
else:
    x  = np.arange(len(valid_apps))
    w  = 0.25
    fig, ax = plt.subplots(figsize=(16, 6))
    for i, (mname, color) in enumerate(zip(models, colors)):
        vals = [pa_f1[a].get(mname) or 0 for a in valid_apps]
        ax.bar(x + (i - 1) * w, vals, w, label=mname, color=color, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(valid_apps, rotation=40, ha="right", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_title("Per-Appliance F1 — 3 모델 비교 (EXP1 val)", fontsize=13)
    ax.set_ylabel("F1")
    ax.legend(fontsize=11)
    ax.axhline(0.5, color="gray", linestyle="--", alpha=0.4)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "baseline_per_appliance_f1.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"저장: {RESULTS_DIR}/baseline_per_appliance_f1.png")

### 10-5. CNN_TDA F1_cls (cls 헤드 단독)

`f1_cls`: CNN_TDA 전용 — BCE로 직접 학습한 on/off 분류 헤드 기준 F1.  
seq2point / bert4nilm에는 cls 헤드가 없으므로 직접 비교 불가 (참고용).

In [ ]:
f1c = RESULTS.get("cnn_tda", {}).get("f1_cls")
if f1c:
    f1_cnn  = RESULTS["cnn_tda"]["f1"]
    f1c_cnn = f1c
    print(f"cnn_tda  F1 (watt thr)  = {f1_cnn:.3f}")
    print(f"cnn_tda  F1_cls (BCE)   = {f1c_cnn:.3f}  ← cls 헤드 직접 최적화")
    print()
    print("seq2point / bert4nilm의 F1과 비교할 때는 F1 (watt thr) 컬럼 기준으로 비교하세요.")
else:
    print("cnn_tda 결과 없음")